# 06 - Multi-head attention in PyTorch

Notebook 05 built causal self-attention with one attention head. Here we extend the same operation to several heads, so each head can learn to focus on different relationships between token positions. The implementation uses ordinary PyTorch layers and a small synthetic input; it does not load a dataset or train a language model.

## 1. From one head to multiple heads

For an input **x** with shape `(B, T, d_in)`, the query, key, and value projections produce vectors of width `d_out`. We divide that width into `num_heads` chunks, each of width `head_dim = d_out // num_heads`. Attention is then computed independently for every head and position.

The key constraint is that `d_out` must be divisible by `num_heads`. After attention, the head outputs are concatenated back into width `d_out`, then an output projection lets the heads exchange information.

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(123)

# B=2 sequences, T=4 positions, d_in=6 features per position.
x = torch.randn(2, 4, 6)
d_in = x.shape[-1]
d_out = 8
num_heads = 2
head_dim = d_out // num_heads

assert d_out % num_heads == 0
print("input:", tuple(x.shape))
print(f"output width: {d_out}; heads: {num_heads}; width per head: {head_dim}")

C:\Users\sail_\Documents\Codex\2026-10-03\https-github-com-rasbt-llms-from\outputs\darija-llm-lab\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


input: (2, 4, 6)
output width: 8; heads: 2; width per head: 4


## 2. Split the projected features into heads

A linear layer can project all heads at once. The reshape makes the head axis explicit, and the transpose moves it next to the batch axis. Each head now has tensors shaped `(B, T, head_dim)`, while the full projected tensor has shape `(B, num_heads, T, head_dim)`.

In [2]:
W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

queries = W_query(x).view(x.shape[0], x.shape[1], num_heads, head_dim).transpose(1, 2)
keys = W_key(x).view(x.shape[0], x.shape[1], num_heads, head_dim).transpose(1, 2)
values = W_value(x).view(x.shape[0], x.shape[1], num_heads, head_dim).transpose(1, 2)

print("queries, keys, values:", tuple(queries.shape))  # (B, H, T, head_dim)

queries, keys, values: (2, 2, 4, 4)


## 3. Compute causal attention for every head

For each head, scaled dot products between queries and keys produce scores shaped `(B, H, T, T)`. The causal mask blocks positions to the right of each query position. A softmax over the last axis turns the remaining scores into weights; multiplying by values produces a context vector for each head.

In [3]:
scores = queries @ keys.transpose(-2, -1)
scores = scores / math.sqrt(head_dim)

sequence_length = x.shape[1]
future_mask = torch.triu(torch.ones(sequence_length, sequence_length, dtype=torch.bool), diagonal=1)
scores = scores.masked_fill(future_mask, -torch.inf)
attention_weights = torch.softmax(scores, dim=-1)
head_context = attention_weights @ values

print("attention weights:", tuple(attention_weights.shape))  # (B, H, T, T)
print("head contexts:", tuple(head_context.shape))          # (B, H, T, head_dim)

attention weights: (2, 2, 4, 4)
head contexts: (2, 2, 4, 4)


## 4. Combine the heads

The attention output is transposed back to `(B, T, H, head_dim)`. We make it contiguous before reshaping, because `transpose` changes the tensor's memory layout. Flattening the final two dimensions concatenates the head vectors, and the output projection mixes their information.

In [4]:
combined = head_context.transpose(1, 2).contiguous().view(x.shape[0], sequence_length, d_out)
out_proj = nn.Linear(d_out, d_out, bias=False)
output = out_proj(combined)

print("combined heads:", tuple(combined.shape))  # (B, T, d_out)
print("projected output:", tuple(output.shape))  # (B, T, d_out)

combined heads: (2, 4, 8)
projected output: (2, 4, 8)


## 5. Package the operation as a reusable module

The module below brings the projections, head reshaping, causal mask, attention, concatenation, and output projection together. The mask is registered as a buffer, so it follows the module when moved to a device but is not treated as a trainable parameter. Dropout is optional and defaults to zero for predictable outputs while studying the tensor flow.

In [5]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0, qkv_bias=False):
        super().__init__()
        if d_out % num_heads != 0:
            raise ValueError("d_out must be divisible by num_heads")

        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        future_mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("future_mask", future_mask)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        if sequence_length > self.future_mask.shape[0]:
            raise ValueError("sequence length exceeds the configured context length")

        queries = self.W_query(x).view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        keys = self.W_key(x).view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(batch_size, sequence_length, self.num_heads, self.head_dim).transpose(1, 2)

        scores = queries @ keys.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)
        mask = self.future_mask[:sequence_length, :sequence_length]
        scores = scores.masked_fill(mask, -torch.inf)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context = attention_weights @ values
        context = context.transpose(1, 2).contiguous().view(batch_size, sequence_length, self.d_out)
        return self.out_proj(context)

## 6. Check the module's shapes and causal mask

A forward pass returns one vector of width `d_out` at each input position. We inspect the attention weights separately using the same equations so the causal structure stays visible: weights at future positions must be zero, and each row must sum to one.

In [6]:
attention = MultiHeadCausalSelfAttention(
    d_in=d_in, d_out=d_out, context_length=8, num_heads=num_heads, dropout=0.0
)
module_output = attention(x)

with torch.no_grad():
    q = attention.W_query(x).view(2, 4, num_heads, head_dim).transpose(1, 2)
    k = attention.W_key(x).view(2, 4, num_heads, head_dim).transpose(1, 2)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
    scores = scores.masked_fill(attention.future_mask[:4, :4], -torch.inf)
    weights = torch.softmax(scores, dim=-1)

assert module_output.shape == (2, 4, d_out)
assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)))
assert torch.count_nonzero(weights[..., torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)]) == 0
print("output shape:", tuple(module_output.shape))
print("attention weights shape:", tuple(weights.shape))
print("future-position weights:", int(torch.count_nonzero(weights[..., torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)])))

output shape: (2, 4, 8)
attention weights shape: (2, 2, 4, 4)
future-position weights: 0


## Key idea

Multi-head attention applies causal self-attention several times in parallel to different learned feature subspaces. The head axis is folded into the batch-like dimensions during the matrix operations, then the outputs are joined and projected back to the model width. The next model-building step is to place this attention layer inside a Transformer block with residual connections, normalization, and a feed-forward network.